# 빠알리어 TTS 생성기 (Indic Parler TTS 산스크리트어)

SuttaLog2 앱용 고품질 빠알리어 음성 파일 생성

**사용법**: Runtime → Run all 실행 후 마지막 셀에서 zip 다운로드

In [ ]:
# 1단계: 패키지 설치
!pip install -q git+https://github.com/ai4bharat/indic-parler-tts torch torchaudio soundfile numpy

In [ ]:
# 2단계: 모델 로드
import torch
from parler_tts import ParlerTTSForConditionalGeneration
from transformers import AutoTokenizer
import soundfile as sf
import os, re, json

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

model_name = 'ai4bharat/indic-parler-tts'
model = ParlerTTSForConditionalGeneration.from_pretrained(model_name).to(device)
tokenizer = AutoTokenizer.from_pretrained(model_name)
print('모델 로드 완료!')

In [ ]:
# 3단계: TTS 텍스트 목록 (앱에서 추출한 528개)
# tts-texts.json을 업로드하거나 아래에 직접 붙여넣기

# 방법 A: 파일 업로드
from google.colab import files
try:
    uploaded = files.upload()  # tts-texts.json 업로드
    texts = json.loads(list(uploaded.values())[0].decode('utf-8'))
    print(f'{len(texts)}개 텍스트 로드')
except:
    print('업로드 실패 - 아래 셀에서 직접 입력하세요')
    texts = []

In [ ]:
# 4단계: 음성 생성
os.makedirs('audio', exist_ok=True)

def text_to_filename(text):
    """텍스트 → 안전한 파일명 변환"""
    # 소문자 + 특수문자 제거 + 공백→하이픈
    name = text.lower().strip()
    name = re.sub(r'[\n\r]', ' ', name)
    name = re.sub(r'[^a-zāīūṭḍṇṅñṃḷ0-9\s\-]', '', name)
    name = re.sub(r'\s+', '-', name.strip())
    name = name[:80]  # 파일명 길이 제한
    return name

# 산스크리트어 음성 설명 (남성, 차분한 톤)
description = 'A male speaker speaks clearly and slowly in Sanskrit with a calm and steady tone.'
desc_ids = tokenizer(description, return_tensors='pt').input_ids.to(device)

total = len(texts)
errors = []

for i, text in enumerate(texts):
    fname = text_to_filename(text)
    outpath = f'audio/{fname}.wav'

    if os.path.exists(outpath):
        print(f'[{i+1}/{total}] 건너뛰기 (이미 존재): {fname}')
        continue

    try:
        # 줄바꿈을 공백으로 변환
        clean_text = text.replace('\n', ' ').strip()
        input_ids = tokenizer(clean_text, return_tensors='pt').input_ids.to(device)

        with torch.no_grad():
            generation = model.generate(
                input_ids=input_ids,
                prompt_input_ids=desc_ids,
                max_new_tokens=500,
            )

        audio = generation.cpu().numpy().squeeze()
        sf.write(outpath, audio, samplerate=22050)
        print(f'[{i+1}/{total}] {fname} ({len(audio)/22050:.1f}s)')

    except Exception as e:
        errors.append((text, str(e)))
        print(f'[{i+1}/{total}] 오류: {fname} - {e}')

print(f'\n완료! 성공: {total - len(errors)}, 오류: {len(errors)}')
if errors:
    print('오류 목록:')
    for t, e in errors:
        print(f'  {t[:40]}... → {e}')

In [ ]:
# 5단계: WAV → MP3 변환 (용량 절약)
!pip install -q pydub
!apt-get -qq install ffmpeg

from pydub import AudioSegment
import glob

os.makedirs('audio_mp3', exist_ok=True)

wav_files = glob.glob('audio/*.wav')
for wf in wav_files:
    mp3f = wf.replace('audio/', 'audio_mp3/').replace('.wav', '.mp3')
    if not os.path.exists(mp3f):
        audio = AudioSegment.from_wav(wf)
        audio.export(mp3f, format='mp3', bitrate='64k')

print(f'MP3 변환 완료: {len(wav_files)}개')

In [ ]:
# 6단계: 파일명 매핑 JSON 생성 + ZIP 다운로드
import zipfile

# 텍스트 → 파일명 매핑
mapping = {}
for text in texts:
    fname = text_to_filename(text)
    mp3path = f'audio_mp3/{fname}.mp3'
    if os.path.exists(mp3path):
        mapping[text] = f'{fname}.mp3'

with open('audio_mp3/manifest.json', 'w', encoding='utf-8') as f:
    json.dump(mapping, f, ensure_ascii=False, indent=2)

print(f'매핑 완료: {len(mapping)}개')

# ZIP 생성
with zipfile.ZipFile('pali-audio.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, fnames in os.walk('audio_mp3'):
        for fn in fnames:
            filepath = os.path.join(root, fn)
            zf.write(filepath, fn)  # 플랫 구조로 저장

size_mb = os.path.getsize('pali-audio.zip') / (1024*1024)
print(f'pali-audio.zip 생성 ({size_mb:.1f} MB)')

# 다운로드
files.download('pali-audio.zip')